# 🎙️ RVC Studio — Colab launcher

A full voice-conversion **app** instead of a stack of cells.

**What you get over the classic RVC notebook**

| | Classic notebook | RVC Studio |
|---|---|---|
| Interface | run cells top to bottom | single web UI, tabs, drag & drop |
| Files | one at a time | **batch** — drop a whole folder |
| Pitch | guess −12 / 0 / +12 | **measured** from your voice vs the model |
| Long audio | breaks / no preview past 4 min | auto-chunked with crossfade, any length |
| Progress | a spinner | live % + logs, cancellable, survives reload |
| Input check | none | clipping / silence / level report *before* you burn GPU time |
| Training | separate notebook | dataset builder with quality score + auto hyper-params |
| Output | one download cell | library with player, re-download, wav/mp3/flac/ogg |
| Reuse | Colab only | same code runs locally, as a CLI, or as a REST API |

Runtime → Change runtime type → **GPU** for real RVC inference, then run the two cells below.

## Step 1 · Install & launch

Takes ~2 minutes the first time. Leave this cell running — it hosts the app.

In [ ]:
#@title Install and start RVC Studio { display-mode: "form" }
REPO = "https://github.com/husseinessamhussein9-ai/hussein"  #@param {type:"string"}
BRANCH = "arena/01a00a0d-hussein"  #@param {type:"string"}
INSTALL_RVC_ENGINE = True  #@param {type:"boolean"}

import os, sys, subprocess, threading, time

if not os.path.exists('/content/hussein'):
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,REPO,'/content/hussein'], check=True)
os.chdir('/content/hussein')

print('Installing dependencies…')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)

if INSTALL_RVC_ENGINE:
    # Real RVC weights + runtime (GPU). Skipped automatically if it is unavailable;
    # the studio then falls back to its portable DSP engine.
    subprocess.run([sys.executable,'-m','pip','install','-q','rvc-python'], check=False)

import rvc_studio.engine as E
print('Engine:', E.engine_info())

def _serve():
    subprocess.run([sys.executable,'-m','rvc_studio.cli','serve','--port','7860'])
threading.Thread(target=_serve, daemon=True).start()
time.sleep(6)

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(7860, height=900)
except Exception:
    print('Open http://localhost:7860')


## Step 2 · Optional — command line

Everything the UI does is scriptable. Handy for large batches.

In [ ]:
#@title Batch convert a folder { display-mode: "form" }
INPUT_FOLDER = "/content/drive/MyDrive/audio_in"  #@param {type:"string"}
MODEL = ""  #@param {type:"string"}
PITCH = 0  #@param {type:"slider", min:-24, max:24, step:1}
AUTO_PITCH = True  #@param {type:"boolean"}
FORMAT = "wav"  #@param ["wav","mp3","flac","ogg"]

cmd = ['python','-m','rvc_studio.cli','batch',INPUT_FOLDER,'-m',MODEL,
       '-p',str(PITCH),'--format',FORMAT]
if AUTO_PITCH: cmd.append('--auto-pitch')
!{' '.join(cmd)}


In [ ]:
#@title Train a voice from raw recordings { display-mode: "form" }
DATASET_NAME = "myvoice"  #@param {type:"string"}
RAW_FOLDER = "/content/drive/MyDrive/raw_voice"  #@param {type:"string"}
EPOCHS = 0  #@param {type:"integer"}

!python -m rvc_studio.cli dataset build {DATASET_NAME} {RAW_FOLDER}/*
# EPOCHS=0 lets the studio pick based on how much material you actually have
extra = f'--epochs {EPOCHS}' if EPOCHS else ''
!python -m rvc_studio.cli train {DATASET_NAME} --dataset {DATASET_NAME} {extra}


---
### Notes
* Without a GPU RVC runtime the studio still runs, using its built-in DSP engine — it transforms the voice (pitch + formants + timbre) but is **not** a trained clone. The UI always tells you which engine produced a file.
* Models are stored in `data/models/`; mount Drive and symlink that folder to keep them between sessions.
* Only clone voices you have permission to use.